<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/17_meta_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json, numpy as np, pandas as pd
from tqdm import tqdm
BASE = '/content/drive/MyDrive/longevity_project'
kd = json.load(open(f'{BASE}/data/processed/known_drugs.json'))
compounds = pd.read_csv(f'{BASE}/data/processed/compounds.csv', index_col='name')
sigpca = pd.read_parquet(f'{BASE}/data/processed/signature_pca.parquet')
fp = pd.read_parquet(f'{BASE}/data/processed/ecfp4.parquet')
T = pd.read_parquet(f'{BASE}/data/processed/targets.parquet')
baseline = pd.read_parquet(f'{BASE}/results/hallmark_matrix_baseline.parquet')
ml = pd.read_parquet(f'{BASE}/results/ml_scores_v2.parquet')
combined = pd.read_parquet(f'{BASE}/results/hallmark_scores_combined.parquet')
holdout = json.load(open(f'{BASE}/data/processed/holdout.json'))
background = json.load(open(f'{BASE}/data/processed/background_negatives.json'))
decisions = json.load(open(f'{BASE}/results/ml_decisions.json'))
kind, feat = decisions['setup'].split('|')
HALLMARKS = list(kd['by_hallmark'])
print('ML setup from Session 13:', kind, 'with', feat, 'features')

from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def similarity(a, b):
    if not (compounds.loc[a, 'has_structure'] and compounds.loc[b, 'has_structure']):
        return 0.0
    x, y = fp.loc[a].values.astype(bool), fp.loc[b].values.astype(bool)
    return (x & y).sum() / (x | y).sum()

def chem_groups(drugs, cutoff=0.6):
    groups = []
    for d in drugs:
        for g in groups:
            if any(similarity(d, m) >= cutoff for m in g):
                g.append(d)
                break
        else:
            groups.append([d])
    return groups

def training_set(hm):
    k = kd['by_hallmark'][hm]
    pos = [d for d in k['helpful'] if d in compounds.index and d not in holdout[hm]]
    neg = [d for d in k['harmful'] if d in compounds.index] + background
    return pos, neg

if feat == 'all':
    has = compounds['has_structure'].values.astype(bool)
    fp_small = np.zeros((len(fp), 50), dtype='float32')
    fp_small[has] = TruncatedSVD(n_components=50, random_state=0).fit_transform(fp.values[has].astype('float32'))
    fpf = pd.DataFrame(fp_small, index=fp.index, columns=[f'fpsvd_{i}' for i in range(50)])
    fpf['has_structure'] = has.astype('float32')
    X = pd.concat([sigpca, fpf, T.astype('float32')], axis=1)
else:
    X = sigpca

def fit(pos, neg, seed=0):
    if kind == 'logistic':
        model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight='balanced', max_iter=3000))
    else:
        model = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight='balanced_subsample',
                                       n_jobs=-1, random_state=seed)
    y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
    return model.fit(X.loc[pos + neg].values, y)

Mounted at /content/drive
ML setup from Session 13: logistic with signatures features


In [2]:
rng = np.random.default_rng(7)
oof = []
for hm in tqdm(HALLMARKS):
    pos, neg = training_set(hm)
    groups = chem_groups(pos)
    pos_fold = {d: i % 5 for i, g in enumerate(rng.permutation(len(groups))) for d in groups[g]}
    neg_fold = dict(zip(neg, rng.integers(0, 5, len(neg))))
    fold_of = {**neg_fold, **pos_fold}
    for f in range(5):
        test = [d for d in pos + neg if fold_of[d] == f]
        train_pos = [d for d in pos if fold_of[d] != f]
        train_neg = [d for d in neg if fold_of[d] != f]
        if not test or not train_pos:
            continue
        s = pd.Series(fit(train_pos, train_neg).predict_proba(X.values)[:, 1], index=X.index)
        pct = s.drop(train_pos + train_neg).rank(pct=True) * 100
        oof += [{'hallmark': hm, 'drug': d, 'label': int(d in pos), 'ml_pct': pct[d],
                 'baseline_pct': baseline.loc[d, f'{hm}_pct']} for d in test]

oof = pd.DataFrame(oof)
print(oof.groupby(['hallmark', 'label']).size().unstack())

100%|██████████| 6/6 [00:07<00:00,  1.26s/it]

label               0  1
hallmark                
autophagy         300  5
inflammation      304  9
mitochondria      302  3
nutrient_sensing  300  6
proteostasis      309  3
senescence        306  5


In [3]:
meta = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000))
meta.fit(oof[['baseline_pct', 'ml_pct']].values, oof['label'].values)

w = meta.named_steps['logisticregression'].coef_[0]
print(f'Weight on gene-list score: {w[0]:.2f}')
print(f'Weight on ML score:        {w[1]:.2f}')

Weight on gene-list score: 0.23
Weight on ML score:        0.09


In [4]:
final = pd.DataFrame(index=compounds.index)
for hm in HALLMARKS:
    inputs = np.c_[baseline[f'{hm}_pct'].reindex(final.index), ml[f'{hm}_pct'].reindex(final.index)]
    p = pd.Series(meta.predict_proba(inputs)[:, 1], index=final.index)
    final[f'{hm}_baseline'] = baseline[f'{hm}_pct'].reindex(final.index)
    final[f'{hm}_ml'] = ml[f'{hm}_pct'].reindex(final.index)
    final[f'{hm}_ml_sd'] = ml[f'{hm}_sd'].reindex(final.index)
    final[f'{hm}_meta'] = (p.rank(pct=True) * 100)
final['is_named'] = compounds['is_named']
final.round(1).to_parquet(f'{BASE}/results/final_scores.parquet')
print(final.shape)

(15542, 25)


In [5]:
rows = []
for hm, drugs in holdout.items():
    for d in drugs:
        rows.append({'hallmark': hm, 'drug': d,
                     'baseline': final.loc[d, f'{hm}_baseline'], 'ml': final.loc[d, f'{hm}_ml'],
                     'ml_uncertainty': final.loc[d, f'{hm}_ml_sd'],
                     'session13_combined': combined.loc[d, f'{hm}_pct'], 'meta': final.loc[d, f'{hm}_meta']})

test = pd.DataFrame(rows).round(1)
print(test.to_string(index=False))
print('\nAverage percentile of held-out drugs:')
print(test[['baseline', 'ml', 'session13_combined', 'meta']].mean().round(1))
test.to_csv(f'{BASE}/results/holdout_test.csv', index=False)

        hallmark          drug  baseline   ml  ml_uncertainty  session13_combined  meta
      senescence      apigenin      57.8 62.5            21.0                57.8  62.9
nutrient_sensing    ku-0063794      99.7 96.4             4.4                96.4  99.9
nutrient_sensing   resveratrol      32.2 63.7            14.2                63.7  38.0
    mitochondria rosiglitazone      92.2 28.3            13.2                92.2  82.4
       autophagy         pp242      97.3 88.6             9.8                97.3  99.6
    inflammation  parthenolide       7.2 86.6            13.8                 7.2  22.4
    inflammation      curcumin       0.2 11.8            25.7                 0.2   0.5
    inflammation       aspirin      48.3 24.8            20.5                48.3  37.8

Average percentile of held-out drugs:
baseline              54.4
ml                    57.8
session13_combined    57.9
meta                  55.4
dtype: float64


In [6]:
dos = pd.read_parquet(f'{BASE}/results/dossiers.parquet')
dos['meta_pct'] = [final.loc[d, f'{h}_meta'] for d, h in zip(dos['drug'], dos['hallmark'])]
dos['ml_uncertainty'] = [final.loc[d, f'{h}_ml_sd'] for d, h in zip(dos['drug'], dos['hallmark'])]
dos['tier'] = np.select(
    [dos['stress_concern'], dos['confidence'] == 'High', dos['confidence'] == 'Medium'],
    ['4 stress flag', '1 High evidence', '2 Medium evidence'], '3 Low evidence')
ranked = dos.sort_values(['tier', 'meta_pct'], ascending=[True, False])
ranked.to_csv(f'{BASE}/results/final_candidates.csv', index=False)

print(ranked[['drug', 'hallmark', 'tier', 'meta_pct', 'ml_uncertainty', 'n_papers']].head(20).to_string(index=False))

multi = ranked[ranked['tier'] < '3'].groupby('drug')['hallmark'].apply(list)
print('\nMedium or High evidence on two or more hallmarks:')
for d, hms in multi[multi.str.len() >= 2].items():
    print(f'  {d}: {", ".join(hms)}')

                drug         hallmark              tier  meta_pct  ml_uncertainty  n_papers
         kenpaullone     mitochondria   1 High evidence 99.974263            11.8         7
       necrostatin-1     inflammation 2 Medium evidence 98.745335            16.9         7
           as-605240       senescence 2 Medium evidence 90.953545            19.6         7
         gsk-1059615 nutrient_sensing    3 Low evidence 99.987132             1.7         2
           k784-3131     proteostasis    3 Low evidence 99.942092             6.8         0
            sib-1893 nutrient_sensing    3 Low evidence 99.909922             0.8         4
             tgx-115 nutrient_sensing    3 Low evidence 99.871316             0.4         0
   aminopurvalanol-a     proteostasis    3 Low evidence 99.790889            18.0         0
orthothymotinic-acid     mitochondria    3 Low evidence 99.732982             7.7         0
            su-11652     proteostasis    3 Low evidence 99.015571            23.

In [7]:
avg = test[['baseline', 'ml', 'meta']].mean()
tiers = ranked['tier'].value_counts()
print(f"Built a pooled logistic-regression meta-model combining gene-list and ML percentiles, trained on out-of-fold "
      f"ML scores (weights: gene list {w[0]:.2f}, ML {w[1]:.2f}). Held-out test on {len(test)} known helpful drugs "
      f"never used in training or tuning: average percentile {avg['baseline']:.0f} (gene list), {avg['ml']:.0f} (ML), "
      f"{avg['meta']:.0f} (meta-model). Ranked {len(ranked)} shortlisted drug-hallmark pairs by AI evidence tier, then "
      f"meta score: {tiers.get('1 High evidence', 0)} High, {tiers.get('2 Medium evidence', 0)} Medium, "
      f"{tiers.get('4 stress flag', 0)} stress-flagged.")

Built a pooled logistic-regression meta-model combining gene-list and ML percentiles, trained on out-of-fold ML scores (weights: gene list 0.23, ML 0.09). Held-out test on 8 known helpful drugs never used in training or tuning: average percentile 54 (gene list), 58 (ML), 55 (meta-model). Ranked 90 shortlisted drug-hallmark pairs by AI evidence tier, then meta score: 1 High, 2 Medium, 68 stress-flagged.
